In [0]:
%run ../../config/metadata_setup

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, DecimalType
from pyspark.sql.functions import col, lit, filter, coalesce, sha2, concat_ws
from datetime import datetime
import os

In [0]:
class SalesTargetsStreaming:
    def __init__(self, spark):
        self.spark = spark
        self.catalog_name = CONFIG['catalog']['name']
        self.schema_name = CONFIG['catalog']['schema']

        # Paths
        self.source_path = CONFIG['paths']['raw_data']
        self.schema_path = CONFIG['paths']['schemas']
        self.checkpoint_path = CONFIG['paths']['checkpoints']

        # sub paths
        self.sub_path = CONFIG['subpaths']['sales_target']

        # table name
        self.table_name = TABLES['bronze']['sales_target']

        # Schema
        self.schema = self._define_schema()

    # -------------------------------
    # Define schema
    # -------------------------------
    def _define_schema(self):
        return StructType([
            StructField('month_of_order_date', StringType(), True),
            StructField('category', StringType(), True),
            StructField('target', StringType(), True)
        ])

    # --------------------------------
    # Read streaming data
    # --------------------------------
    def read_stream(self):
        return (
            self.spark.readStream
            .format('cloudFiles')
            .option('cloudFiles.format', 'csv')
            .option('cloudFiles.schemaLocation', f"{self.schema_path}/_{self.table_name}")
            .option('cloudFiles.schemaEvolutionMode', 'rescue')
            .option("delimiter", ",")
            .schema(self.schema)
            .load(f"{self.source_path}/{self.sub_path}")           
        )    
    
    # --------------------------------
    # header cleaning
    # ---------------------------------
    def clean_header(self, df, header_col:str, header_value:str):
        """
        Removing header row
        """ 
        try:
            return df.filter(col(header_col) != header_value)
        except Exception as e:
            print(f'Error: clean_header: {e}')

    # ----------------------------------
    # add file metadata
    # ----------------------------------
    def add_metadata(self, df):
        return (df.withColumn('ingest_ts', lit(datetime.now()))
                  .withColumn('file_name', col("_metadata.file_name"))
                  .withColumn('file_path', col("_metadata.file_path"))
                  .withColumn('file_modification_time', col("_metadata.file_modification_time"))
                  .withColumn('file_size', col("_metadata.file_size"))
                  .drop("_metadata"))

    # ---------------------------------
    # Added surrogate key
    # ---------------------------------
    def add_surrogate_key(self, df):
        """
        Adding surrogate key
        """        
        return df.withColumn('surrogate_key', sha2(concat_ws('||', *[coalesce(col(c).cast("string"),lit('NULL')) for c in df.columns]), 256))

    # ---------------------------------
    # display table
    # ---------------------------------
    def display_tbl(self):
        return self.spark.sql(f"SELECT * FROM {self.catalog_name}.{self.schema_name}.{self.table_name}")

    # --------------------------------
    # Only for droping table
    #---------------------------------
    def drop_table(self):
        dbutils.fs.rm(f"{self.schema_path}/_{self.table_name}", True)
        dbutils.fs.rm(f"{self.checkpoint_path}/_{self.table_name}", True)
        self.spark.sql(f"DROP TABLE IF EXISTS {self.catalog_name}.{self.schema_name}.{self.table_name}")
    
    # ---------------------------------
    # write to delta table (streaming)
    # ---------------------------------
    def write_stream(self, df):
        try:
            query = (
                df.writeStream
                .format('delta')
                .option('checkpointLocation', f"{self.checkpoint_path}/_{self.table_name}")
                .trigger(availableNow=True)
                .toTable(f'{self.catalog_name}.{self.schema_name}.{self.table_name}')
            )   
            query.awaitTermination()
        except Exception as e:
            print(f'Error: write_stream: {e}')  

    def run(self):
        df = self.read_stream()
        df = self.add_metadata(df)
        df = self.add_surrogate_key(df)
        df = self.clean_header(df, 'category', 'Category')
        self.write_stream(df)   

In [0]:
obj = SalesTargetsStreaming(spark)
obj.run()